# BTK Datathon 2026 — Modelling Notebook

**Competition:** Career Success Score Prediction  
**Metric:** MSE (lower is better)  
**Author:** emircankartal  

This notebook documents the complete modelling pipeline for jury presentation:  
1. Preprocessing decisions (imputation, encoding)  
2. Feature engineering walkthrough (all 3 groups)  
3. 5-fold CV results per model  
4. Ridge meta-learner stacking  
5. SHAP feature importance  
6. Prediction distribution analysis  
7. Final submission validation

In [ ]:
import os, sys
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from src.utils.config import load_config
from src.utils.io import load_data
from src.features.engineering import (
    build_features, impute_fold, build_rank_features_fold,
    compute_text_stats, TARGET_ENCODE_COLS, get_feature_cols,
    TECHNICAL_SKILL_COLS, SOFT_SKILL_COLS,
)
from src.features.target_encoding import smoothed_target_encode, frequency_encode
from src.nlp.text_features import get_sentence_embeddings, fold_svd_reduce, embedding_to_dataframe

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
cfg = load_config()
SEED = 42
print('Setup complete')

## 1. Data Loading & Overview

In [ ]:
train_raw = load_data(cfg['data']['train_file'])
test_raw  = load_data(cfg['data']['test_file'])
y = train_raw[cfg['data']['target_col']].values.astype(np.float32)

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
print(f'\nTarget stats:')
print(train_raw['career_success_score'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Target Distribution: career_success_score')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Count')

axes[1].boxplot(y, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot: career_success_score')
axes[1].set_xlabel('Score')
plt.tight_layout()
plt.savefig('../../outputs/plots/target_distribution.png', bbox_inches='tight')
plt.show()

## 2. Preprocessing Pipeline

### 2.1 Missing Value Strategy

| Column | Strategy | Rationale |
|--------|----------|----------|
| `github_avg_stars`, `open_source_contribution_count` | Fill 0 + `has_github` flag | No GitHub account → 0 activity, not unknown |
| `internship_duration_months`, `hr_interview_score`, `linkedin_profile_score`, `portfolio_score`, `english_exam_score` | Fold-median + missing flag | Random missingness, fold-safe to prevent leakage |

### 2.2 Encoding
- `department`, `target_role`: Smoothed K-fold target encoding (α=10) + frequency encoding — prevents leakage
- `university_tier`: Ordinal (Tier 1=4, Tier 2=3, Tier 3=2, Tier 4=1) — monotonic with target

In [ ]:
# Missing value audit
missing = train_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, max(3, len(missing)*0.4)))
bars = ax.barh(missing.index, missing.values, color='coral', edgecolor='white')
ax.set_title('Missing Values in Training Set')
ax.set_xlabel('Count')
for bar, val in zip(bars, missing.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val} ({100*val/len(train_raw):.1f}%)', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../../outputs/plots/missing_values.png', bbox_inches='tight')
plt.show()
print(missing)

## 3. Feature Engineering

### Group A — Technical Depth & Breadth
9 skill scores are highly intercorrelated (r≈0.5-0.6). Features capture:
- `tech_skill_mean/std/max/min/range` — level and consistency  
- `skill_ceiling_count` (>80) — mastery breadth  
- `skill_floor_count` (<50) — weakness count  
- `technical_breadth` (>60/9) — solid-skill proportion  
- `top3_skills_mean` — ceiling of ability  
- `cloud_devops_mean` — cluster from highest-scoring roles  

### Group B — Skill-Role Alignment
- `skill_role_match` — mean of role-relevant skills  
- `skill_role_gap` — specialist premium vs overall average  
- `role_value_score` — empirical career premium (EDA: 6.4pt spread)  

### Group C — Project-Centric Interactions
`project_quality_score` is the king feature (r=0.541). Used as multiplicative amplifier:
- `project_x_interview`, `project_x_portfolio`, `project_x_experience`
- `real_world_focus`, `competitive_strength`, `activities_total`, `projects_per_year`

### Group D — Fold-Safe Rank Features (fitted inside each fold)
- `project_rank_in_dept` — percentile rank within department  
- `tech_rank_in_role` — percentile rank within target role

In [ ]:
# Feature engineering preview
train_fe = build_features(train_raw)
text_stats = compute_text_stats(train_raw['mentor_feedback_text'])
train_fe = pd.concat([train_fe.reset_index(drop=True), text_stats.reset_index(drop=True)], axis=1)

new_feats = [
    'skill_ceiling_count', 'skill_floor_count', 'technical_breadth', 'top3_skills_mean',
    'cloud_devops_mean', 'skill_role_match', 'skill_role_gap', 'role_value_score',
    'total_experience', 'real_world_focus', 'competitive_strength',
    'project_x_interview', 'project_x_portfolio', 'project_x_experience',
    'text_pos_count', 'text_neg_count', 'text_sentiment_proxy',
]
new_feats = [f for f in new_feats if f in train_fe.columns]

# Correlations with target
corrs = pd.concat([train_fe[new_feats], train_raw[['career_success_score']]], axis=1).corr()['career_success_score'].drop('career_success_score').sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['coral' if v < 0 else 'steelblue' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('New Feature Correlations with career_success_score')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.savefig('../../outputs/plots/new_feature_correlations.png', bbox_inches='tight')
plt.show()
print(corrs.round(3))

## 4. Model Training — 5-Fold CV Results

In [ ]:
# Load cached embeddings
train_emb = get_sentence_embeddings(
    train_raw['mentor_feedback_text'],
    model_name=cfg['nlp']['embedding_model'],
    cache_path=cfg['nlp']['cache_train'],
)
test_emb = get_sentence_embeddings(
    test_raw['mentor_feedback_text'],
    model_name=cfg['nlp']['embedding_model'],
    cache_path=cfg['nlp']['cache_test'],
)
print(f'Train embeddings: {train_emb.shape}  |  Test embeddings: {test_emb.shape}')

test_fe = build_features(test_raw)
test_text = compute_text_stats(test_raw['mentor_feedback_text'])
test_fe = pd.concat([test_fe.reset_index(drop=True), test_text.reset_index(drop=True)], axis=1)

base_feat_cols = get_feature_cols(train_fe)
n_splits = cfg['cross_validation']['n_splits']
svd_n = cfg['nlp']['svd_components']
kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

In [ ]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import ExtraTreesRegressor

MODEL_NAMES = ['lgbm', 'xgb', 'catboost', 'extra_trees']
oof = {m: np.zeros(len(train_fe)) for m in MODEL_NAMES}
test_preds = {m: np.zeros(len(test_fe)) for m in MODEL_NAMES}
fold_mse_table = {m: [] for m in MODEL_NAMES}
best_models = {m: None for m in MODEL_NAMES}  # Store last fold model for SHAP
last_X_val_final = None
last_y_val = None
last_feat_names = None

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_fe), 1):
    print(f'\n--- Fold {fold}/{n_splits} ---')
    X_tr_raw  = train_fe.iloc[tr_idx].copy()
    X_val_raw = train_fe.iloc[val_idx].copy()
    y_tr  = pd.Series(y[tr_idx], index=X_tr_raw.index)
    y_val = y[val_idx]

    X_tr_imp, X_val_imp, X_test_imp = impute_fold(
        X_tr_raw[base_feat_cols], X_val_raw[base_feat_cols], test_fe[base_feat_cols]
    )
    X_tr_imp, X_val_imp, X_test_imp = build_rank_features_fold(X_tr_imp, X_val_imp, X_test_imp)

    X_tr_enc, X_val_enc, X_test_enc = smoothed_target_encode(
        X_tr_imp, X_val_imp, y_tr, TARGET_ENCODE_COLS, X_test=X_test_imp
    )
    X_tr_enc, X_val_enc, X_test_enc = frequency_encode(
        X_tr_enc, X_val_enc, TARGET_ENCODE_COLS, X_test=X_test_enc
    )

    gbdt_drop = [c for c in TARGET_ENCODE_COLS if c in X_tr_enc.columns]
    X_tr_g   = X_tr_enc.drop(columns=gbdt_drop, errors='ignore')
    X_val_g  = X_val_enc.drop(columns=gbdt_drop, errors='ignore')
    X_test_g = X_test_enc.drop(columns=gbdt_drop, errors='ignore')

    emb_tr, emb_val, emb_test, _ = fold_svd_reduce(train_emb[tr_idx], train_emb[val_idx], svd_n, test_emb)
    X_tr_f   = np.hstack([X_tr_g.values,   embedding_to_dataframe(emb_tr).values])
    X_val_f  = np.hstack([X_val_g.values,  embedding_to_dataframe(emb_val).values])
    X_test_f = np.hstack([X_test_g.values, embedding_to_dataframe(emb_test).values])
    feat_names = list(X_tr_g.columns) + [f'emb_{i}' for i in range(svd_n)]

    # LightGBM
    lgbm_p = {k: v for k, v in cfg['models']['lgbm'].items()}
    lgbm_p['random_state'] = SEED
    lgbm = LGBMRegressor(**lgbm_p)
    lgbm.fit(X_tr_f, y_tr.values, eval_set=[(X_val_f, y_val)],
             callbacks=[early_stopping(100, verbose=False), log_evaluation(-1)])
    p = lgbm.predict(X_val_f); oof['lgbm'][val_idx] = p; fold_mse_table['lgbm'].append(mean_squared_error(y_val, p))
    test_preds['lgbm'] += lgbm.predict(X_test_f) / n_splits

    # XGBoost
    xgb_p = {k: v for k, v in cfg['models']['xgb'].items()}; xgb_p['seed'] = SEED
    xgb = XGBRegressor(**xgb_p, early_stopping_rounds=100, eval_metric='rmse')
    xgb.fit(X_tr_f, y_tr.values, eval_set=[(X_val_f, y_val)], verbose=False)
    p = xgb.predict(X_val_f); oof['xgb'][val_idx] = p; fold_mse_table['xgb'].append(mean_squared_error(y_val, p))
    test_preds['xgb'] += xgb.predict(X_test_f) / n_splits

    # CatBoost
    cb_p = {k: v for k, v in cfg['models']['catboost'].items()}; cb_p['random_seed'] = SEED
    cat_present = [c for c in TARGET_ENCODE_COLS if c in X_tr_enc.columns]
    X_tr_cat = pd.concat([X_tr_enc[cat_present].reset_index(drop=True),
                           X_tr_g.reset_index(drop=True),
                           embedding_to_dataframe(emb_tr).reset_index(drop=True)], axis=1)
    X_val_cat = pd.concat([X_val_enc[cat_present].reset_index(drop=True),
                            X_val_g.reset_index(drop=True),
                            embedding_to_dataframe(emb_val).reset_index(drop=True)], axis=1)
    X_test_cat = pd.concat([X_test_enc[cat_present].reset_index(drop=True),
                             X_test_g.reset_index(drop=True),
                             embedding_to_dataframe(emb_test).reset_index(drop=True)], axis=1)
    for df in [X_tr_cat, X_val_cat, X_test_cat]:
        for c in cat_present: df[c] = df[c].fillna('unknown').astype(str)
    cb = CatBoostRegressor(**cb_p)
    cb.fit(Pool(X_tr_cat, y_tr.values, cat_features=cat_present),
           eval_set=Pool(X_val_cat, y_val, cat_features=cat_present), verbose=False)
    p = cb.predict(X_val_cat); oof['catboost'][val_idx] = p; fold_mse_table['catboost'].append(mean_squared_error(y_val, p))
    test_preds['catboost'] += cb.predict(X_test_cat) / n_splits

    # ExtraTrees
    et_p = {k: v for k, v in cfg['models']['extra_trees'].items()}; et_p['random_state'] = SEED
    et = ExtraTreesRegressor(**et_p)
    et.fit(X_tr_f, y_tr.values)
    p = et.predict(X_val_f); oof['extra_trees'][val_idx] = p; fold_mse_table['extra_trees'].append(mean_squared_error(y_val, p))
    test_preds['extra_trees'] += et.predict(X_test_f) / n_splits

    print(f'  LGBM={fold_mse_table["lgbm"][-1]:.2f}  XGB={fold_mse_table["xgb"][-1]:.2f}  '
          f'CAT={fold_mse_table["catboost"][-1]:.2f}  ET={fold_mse_table["extra_trees"][-1]:.2f}')

    if fold == n_splits:
        best_models['lgbm'] = lgbm
        best_models['xgb']  = xgb
        last_X_val_final = X_val_f
        last_y_val = y_val
        last_feat_names = feat_names

print('\nDone!')

## 5. CV Results Summary

In [ ]:
print('\n=== OOF Results ===')
oof_summary = {}
for name in MODEL_NAMES:
    mse = mean_squared_error(y, oof[name])
    oof_summary[name] = mse
    print(f'  {name:<15} OOF MSE={mse:.4f}  RMSE={np.sqrt(mse):.4f}')

# Per-fold bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(n_splits)
width = 0.2
colors_m = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
for i, (name, color) in enumerate(zip(MODEL_NAMES, colors_m)):
    fmses = fold_mse_table[name]
    ax.bar(x + i*width, fmses, width, label=name, color=color, alpha=0.8, edgecolor='white')
ax.set_xlabel('Fold')
ax.set_ylabel('MSE')
ax.set_title('Per-Fold MSE by Model')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels([f'Fold {i+1}' for i in range(n_splits)])
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/plots/fold_mse_comparison.png', bbox_inches='tight')
plt.show()

# OOF MSE comparison
fig, ax = plt.subplots(figsize=(8, 4))
names = list(oof_summary.keys())
vals  = list(oof_summary.values())
colors_b = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
bars = ax.bar(names, vals, color=colors_b, edgecolor='white', alpha=0.85)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:.2f}', ha='center', va='bottom', fontweight='bold')
ax.set_title('OOF MSE — Model Comparison')
ax.set_ylabel('MSE')
ax.set_ylim(min(vals)*0.95, max(vals)*1.08)
plt.tight_layout()
plt.savefig('../../outputs/plots/oof_mse_comparison.png', bbox_inches='tight')
plt.show()

## 6. Ridge Meta-Learner Stacking

In [ ]:
oof_stack  = np.column_stack([oof[m] for m in MODEL_NAMES])
test_stack = np.column_stack([test_preds[m] for m in MODEL_NAMES])

best_alpha, best_mse = 1.0, np.inf
for alpha in cfg['ensemble']['ridge_alphas']:
    r = Ridge(alpha=alpha).fit(oof_stack, y)
    mse = mean_squared_error(y, r.predict(oof_stack))
    if mse < best_mse:
        best_mse, best_alpha = mse, alpha

ridge = Ridge(alpha=best_alpha).fit(oof_stack, y)
stacked_oof   = ridge.predict(oof_stack)
stacked_test  = np.clip(ridge.predict(test_stack), 0, 100)
stacked_mse   = mean_squared_error(y, stacked_oof)

print(f'Stacked OOF MSE: {stacked_mse:.4f}  (RMSE: {np.sqrt(stacked_mse):.4f})')
print(f'Ridge alpha={best_alpha}')
print(f'Ridge weights: {dict(zip(MODEL_NAMES, ridge.coef_.round(4)))}')
print(f'Ridge intercept: {ridge.intercept_:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
weights = ridge.coef_
colors_w = ['steelblue' if w > 0 else 'coral' for w in weights]
ax.barh(MODEL_NAMES, weights, color=colors_w, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Ridge Stacking Weights (α={best_alpha})')
ax.set_xlabel('Weight')
plt.tight_layout()
plt.savefig('../../outputs/plots/ridge_weights.png', bbox_inches='tight')
plt.show()

## 7. SHAP Feature Importance

In [ ]:
try:
    import shap
    shap.initjs()

    explainer = shap.TreeExplainer(best_models['lgbm'])
    shap_values = explainer.shap_values(last_X_val_final)

    # Top-20 features
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top20_idx = np.argsort(mean_abs_shap)[-20:][::-1]
    top20_names = [last_feat_names[i] for i in top20_idx]
    top20_vals  = mean_abs_shap[top20_idx]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top20_names[::-1], top20_vals[::-1], color='steelblue', edgecolor='white')
    ax.set_title('SHAP Feature Importance — Top 20 (LightGBM, Fold 5)')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig('../../outputs/plots/shap_top20.png', bbox_inches='tight')
    plt.show()
except ImportError:
    print('shap not installed. Run: pip install shap')

## 8. Prediction Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted scatter
axes[0].scatter(y, stacked_oof, alpha=0.3, s=8, color='steelblue')
lims = [min(y.min(), stacked_oof.min()), max(y.max(), stacked_oof.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted (OOF)')
axes[0].set_title(f'Actual vs Predicted (OOF MSE={stacked_mse:.2f})')
axes[0].legend()

# Residual distribution
residuals = stacked_oof - y
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual (Predicted - Actual)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution  (mean={residuals.mean():.3f}, std={residuals.std():.3f})')

plt.tight_layout()
plt.savefig('../../outputs/plots/prediction_analysis.png', bbox_inches='tight')
plt.show()

## 9. Final Submission Validation

In [ ]:
# Load existing submission
import glob
sub_files = sorted(glob.glob('../../data/submissions/*.csv'))
print('Available submissions:')
for f in sub_files:
    print(f'  {f}')

if sub_files:
    sub = pd.read_csv(sub_files[-1])
    print(f'\nSubmission shape: {sub.shape}')
    print(f'Columns: {list(sub.columns)}')
    print(f'\nPrediction stats:')
    print(sub.describe().round(3))
    print(f'\nNull check: {sub.isnull().sum().sum()} nulls')
    print(f'Range check: [{sub.iloc[:,1].min():.3f}, {sub.iloc[:,1].max():.3f}] (should be 0-100)')

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(sub.iloc[:,1], bins=50, color='steelblue', edgecolor='white', alpha=0.8, label='Test predictions')
    ax.hist(y, bins=50, color='coral', edgecolor='white', alpha=0.5, label='Train actuals')
    ax.set_title('Prediction Distribution: Test vs Train')
    ax.set_xlabel('career_success_score')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../../outputs/plots/submission_distribution.png', bbox_inches='tight')
    plt.show()

## 10. Summary Table

In [ ]:
summary_df = pd.DataFrame({
    'Model': MODEL_NAMES + ['Ridge Stack'],
    'OOF MSE': [oof_summary[m] for m in MODEL_NAMES] + [stacked_mse],
    'OOF RMSE': [np.sqrt(oof_summary[m]) for m in MODEL_NAMES] + [np.sqrt(stacked_mse)],
})
summary_df['OOF MSE'] = summary_df['OOF MSE'].round(4)
summary_df['OOF RMSE'] = summary_df['OOF RMSE'].round(4)
summary_df = summary_df.sort_values('OOF MSE')
print(summary_df.to_string(index=False))